# 01_01 - OMIP Energy Market Data Extraction
## 1. Methodology Overview

To analyze energy market resilience, we require historical daily prices from the OMIP platform, encompassing both the Spanish Spot market (SPEL) and Monthly Futures contracts (M+1 to M+6).

**Extraction Architecture:**
Due to the computationally expensive nature of web scraping (which includes ethical delays to prevent server overload and IP blocking), the raw data extraction is intentionally detached from this interactive notebook. 

The main scraping engine (`OMIPScraper`) is modularized and executed via an external Python script located at `src/data/extract_omip_data.py`. This script handles:
1. DOM parsing via `pandas.read_html`.
2. Spoofing HTTP headers to mimic legitimate browser traffic.
3. Forward-Fill (`ffill`) imputation to carry Friday closing prices over non-trading weekends.

### 1.2. Temporal Scope and Reproducibility

The temporal boundaries for the data extraction are dynamically parameterized within the `extract_omip_data.py` script. For this study, the extraction window was configured from **January 1st, 2020, to December 31st, 2024**. 

This specific timeframe was strategically selected to capture a full cycle of market volatility, encompassing:
1. The pre-disruption baseline (early 2020).
2. The severe European energy crisis and price shocks (2021-2022).
3. The subsequent market stabilization and adaptation phase (2023-2024).

*Note for reproducibility:* To update the dataset with recent market data or to investigate different historical periods, researchers can simply adjust the `start_date` and `end_date` variables within the `__main__` execution block of the source script prior to running it.

## 2. Loading the Extracted Dataset

In this section, we load the pre-compiled output generated by our external script to validate its structural integrity before proceeding to the Exploratory Data Analysis (EDA) phase.

In [ ]:
import pandas as pd
from pathlib import Path

raw_data_path = Path("../../data/raw/omip/omip_prices_raw..csv")

# Load and validate the dataset
try:
    omip_raw_df = pd.read_csv(raw_data_path)
    omip_raw_df['Date'] = pd.to_datetime(omip_raw_df['Date'])
    
    print("✅ Raw OMIP dataset successfully loaded.")
    print("-" * 40)
    print(f"Dataset Dimensions: {omip_raw_df.shape[0]} rows, {omip_raw_df.shape[1]} columns")
    print(f"Time Range: {omip_raw_df['Date'].min().date()} to {omip_raw_df['Date'].max().date()}")
    print("-" * 40)
    
except FileNotFoundError:
    print(f"❌ Error: Dataset not found. Please execute 'python src/data/extract_omip_data.py' first.")

✅ Raw OMIP dataset successfully loaded.
----------------------------------------
Dataset Dimensions: 2192 rows, 14 columns
Time Range: 2020-01-01 to 2025-12-31
----------------------------------------


In [3]:
# Display a sample of the extracted features (Spot and Forward curves)
omip_raw_df.head(5)

,Date,Spot_Price_SPEL,Future_M1_Price,Future_M1_OpenInterest,Future_M2_Price,Future_M2_OpenInterest,Future_M3_Price,Future_M3_OpenInterest,Future_M4_Price,Future_M4_OpenInterest,Future_M5_Price,Future_M5_OpenInterest,Future_M6_Price,Future_M6_OpenInterest
0,2020-01-01,35.54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-02,40.00,44.81,1383.0,41.7,1397.0,38.54,10.0,42.11,0.0,47.41,0.0,46.23,0.0
2,2020-01-03,39.51,45.25,1383.0,41.9,1405.0,38.90,10.0,42.50,0.0,47.85,0.0,46.67,0.0
3,2020-01-04,35.67,45.25,1383.0,41.9,1405.0,38.90,10.0,42.50,0.0,47.85,0.0,46.67,0.0
4,2020-01-05,38.18,45.25,1383.0,41.9,1405.0,38.90,10.0,42.50,0.0,47.85,0.0,46.67,0.0
